In [1]:
# D4.1 Complex Method Chaining
# Attempt to write a production quality chain

import pandas as pd
import numpy as np
np.random.seed(42)

suppliers = ["Acme", "GlobalCo", "FastParts", "PrimeMfg", "EastCoast"]
categories = ["Electronics", "Hardware", "Consumables", "Apparel"]
warehouses = ["East", "West", "Central"]

rows = []
for _ in range(200):
    rows.append({
        "po_id": f"PO-{str(np.random.randint(1000, 9999)).zfill(4)}",
        "supplier": np.random.choice(suppliers),
        "category": np.random.choice(categories),
        "warehouse": np.random.choice(warehouses),
        "order_date": pd.Timestamp("2024-01-01") + pd.Timedelta(days=int(np.random.randint(0, 365))),
        "amount": round(np.random.uniform(500, 50000), 2),
        "units": np.random.randint(3, 45),
        "lead_time_days": np.random.randint(3, 45),
        "on_time": np.random.choice([True, False], p=[0.8, 0.2])
    }
    )
df=pd.DataFrame(rows)
print(df.shape)
print(df.head)

(200, 9)
<bound method NDFrame.head of        po_id   supplier     category warehouse order_date    amount  units  \
0    PO-8270   PrimeMfg  Electronics   Central 2024-04-16  39094.70     23   
1    PO-5426  FastParts      Apparel      East 2024-04-09   7571.91      5   
2    PO-7949   GlobalCo      Apparel      East 2024-06-09  15559.99     24   
3    PO-3558   GlobalCo      Apparel   Central 2024-07-08   4985.02      5   
4    PO-6393   PrimeMfg  Electronics      West 2024-08-29  19578.12     28   
..       ...        ...          ...       ...        ...       ...    ...   
195  PO-1525   GlobalCo  Consumables      East 2024-11-20   3922.07     43   
196  PO-4155   GlobalCo  Consumables      West 2024-05-24  19858.54     30   
197  PO-5356       Acme      Apparel      East 2024-10-29  12116.36     12   
198  PO-6724   GlobalCo      Apparel      West 2024-12-26  21828.93     22   
199  PO-7656       Acme      Apparel      West 2024-10-10   2315.24     39   

     lead_time_days  on_

### **Drill 1: Five-step chain -- supplier performance summary**

1. Write a single chain with no intermediate variables that:

- Filters for POs where amount exceeds $5,000
- Adds a quarter column extracted from order_date
- Adds an otd_flag — 1 if on_time is True, 0 if False
- Groups by supplier and quarter and calculates:

Definitions
- total_spend — sum of amount
- avg_lead_time — mean of lead_time_days
- otd_rate — mean of otd_flag × 100 rounded to 1 decimal

Final: 
- Sorts by supplier ascending then total_spend descending

In [2]:
# drill 1
# lead time = order delivery date - order request date ==> total time elapsed between start and receipt of goods
result = (
    df[df["amount"] > 5000]
    .assign(
        order_date = lambda x: pd.to_datetime(x["order_date"]),
        quarter = lambda x: x["order_date"].dt.quarter,
        otd_flag = lambda x: np.where(x["on_time"] == True,1,0)       
    )
    .groupby(by=["supplier", "quarter"])
    .agg(
        total_spend = ("amount", "sum"),
        avg_lead_time=("lead_time_days", "mean"),
        otd_rate = ("otd_flag", lambda x: (x.mean() * 100).round(1))
    )
    .sort_values(["supplier", "total_spend"],
                 ascending=[True, False])
)

print(result)

                   total_spend  avg_lead_time  otd_rate
supplier  quarter                                      
Acme      4          310902.31      19.818182      81.8
          1          260376.96      16.900000     100.0
          3          164208.46      15.285714      71.4
          2          159697.10      26.300000      80.0
EastCoast 3          324792.58      23.777778      66.7
          1          219146.04      21.090909      63.6
          2          169418.87      31.750000     100.0
          4          141560.91      24.166667      50.0
FastParts 4          302773.13      26.000000      72.7
          2          288320.15      26.000000      81.8
          1          266549.84      24.000000      66.7
          3          196825.26      29.125000      87.5
GlobalCo  3          275543.89      25.000000      88.9
          4          264014.46      16.000000      77.8
          2          244458.40      23.900000      70.0
          1          166801.58      21.250000   

## Drill 2: Warehouse Inventory Chain

1. Add inventory_value = amount × units
2. Add value_tier using pd.qcut() — 3 tiers: "low", "mid", "high"
3. Filter for only "high" tier
4. .pivot_table() — warehouse × category, total inventory_value
5. Add warehouse_total using .assign() + .transform()
6. Add pct_of_warehouse — each category's share of warehouse total, rounded to 1 decimal

In [3]:
# Drill 2
newdf = (
    df
    .assign(
        inventory_value = lambda x: (x["amount"] * x["units"]).round(2),
        value_tier = lambda x: pd.qcut(x=x["amount"], q=3, labels=["low", "medium", "high"])
    )
    .loc[lambda x: x["value_tier"] == "high"]
    .pivot_table(
        index="warehouse",
        columns="category",
        values="inventory_value",
        aggfunc="sum"
    )
    .round(2)
    .assign(
        warehouse_total = lambda x: x.sum(axis=1).round(2), # row total across all category columns
        pct_of_warehouse = lambda x: (x["warehouse_total"] / x["warehouse_total"].sum() * 100).round(2) # .sum() is the grand total acros all warehouses
    )
)

print(newdf)

category      Apparel  Consumables  Electronics    Hardware  warehouse_total  \
warehouse                                                                      
Central    1552196.89   6075502.24  12596214.67  5244905.42      25468819.22   
East       6711412.16   2812628.36   3154123.88         NaN      12678164.40   
West       1874860.09   3090133.25   8070422.11  4036623.22      17072038.67   

category   pct_of_warehouse  
warehouse                    
Central               46.12  
East                  22.96  
West                  30.92  


## Drill 3: Validate the Data
- check null values
- check for duplicates
- checl for negative amounts


In [ ]:
# Drill 3

def validate_po_data(df):
    nulls = df.isnull().sum()
    print(f"Null values: {nulls}")
    dupes = df["po_id"].duplicated().sum()
    print(f"Duplicate PO ID's: {dupes}")
    neg = (df["amount"] <= 0).sum()
    print(f"Non-Positive amounts: {neg}")
    return df

# build a 5-step chain using .pipe(validate_po_data)
